# Particle Filter

Das Ziel des hier implementierten Particle Filters ist es, die Position des Smartphones zum Zeitpunkt $k$ zu bestimmen.<br>
Eingabedaten kommen dabei zum einen aus dem Pedestrian dead Reckoning, zum anderen aus der Trilateration der RSSI Daten. Zum fusionieren der beiden Datenquellen kann sowohl ein Kalman-, als auch ein Particle Filter verwendet werden. Letzterer eignet sich aus mehreren Gründen besonders für den gegebenen Anwendungsfall und wird deshalb implementiert:
- Durch die Lokalisierung im Innenbereich eines Gebäudes liegen natürliche Restriktionen des möglichen Pfads vor (Wände). Diese können beim Particle-Filter gut modelliert werden, indem man verhindert, dass die einzelnen Partikel zwischen den Zeitpunkten $k$ und $k-1$ eine Wand kreuzen.
- Die Fehlerverteilung von RSSI-Signalen entspricht aufgrund von Abschattungen und Mehrwegeausbreitung in Innenräumen eher einer PDF (Propability-Density-Function) als einer gausschen Normalverteilung.

## Import statements

In [1]:
import sqlite3
import time

import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from IPython.display import clear_output

import math
import pandas as pd
import numpy as np

rng = np.random.default_rng()

## Read data

In [2]:
conn = sqlite3.connect("../data/emi_nav.db")

run_id =  "R1"

df_steps = pd.read_sql(
    f"""
        SELECT * FROM steps
        WHERE run_id = '{run_id}'
        ORDER BY timestamp_ms
    """, conn)

runs_df   = pd.read_sql('SELECT * FROM runs', conn)

df_rssi = pd.read_sql_query(
    """
    SELECT
        run_id,
        timestamp_ms,
        beacon_name,
        address,
        rssi
    FROM ble_rssi
    WHERE REPLACE(address, ':', '') != 'FBADD6D492CE'
    ORDER BY run_id, timestamp_ms
    """,
    conn
)

df_heading = pd.read_sql(
    f"""
        SELECT * FROM heading
        WHERE run_id = '{run_id}'
        ORDER BY timestamp_ms
    """, conn)

# Beacon-Positionen aus DB
beacon_pos_df = pd.read_sql('SELECT * FROM beacon_positions', conn)
BEACON_POSITIONS = {
    row.beacon_name: (row.x_m, row.y_m, row.floor)
    for row in beacon_pos_df.itertuples()
}

# Door-Positionen + Groundtruth per JOIN aus DB
gt_df = pd.read_sql("""
    SELECT g.run_id, g.tuer_id, g.timestamp_ms,
           d.x_m, d.y_m, d.floor
    FROM groundtruth g
    JOIN door_positions d ON g.tuer_id = d.tuer_id
    WHERE d.x_m IS NOT NULL AND d.y_m IS NOT NULL
""", conn)

df_rssi = df_rssi[df_rssi['beacon_name'].str.startswith('arrive_emi')].copy()
df_rssi = df_rssi[df_rssi['rssi'] <= 0].copy()
df_steps['t_sec'] = (df_steps.timestamp_ms - df_steps.timestamp_ms.min()) / 1000
df_heading['t_sec'] = (df_heading.timestamp_ms - df_heading.timestamp_ms.min()) / 1000
df_rssi['t_sec'] = (df_rssi.timestamp_ms - df_rssi.timestamp_ms.min()) / 1000

In [3]:
map_eg = np.load(r'..\data\floorplans\eg.npy', allow_pickle=True)
map_og1 = np.load(r'..\data\floorplans\og1.npy', allow_pickle=True)

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\floorplans\\eg.npy'

## Kombination von df_steps und df_heading
Bei jedem Schritt wird ein Prediktionsschritt im Filter ausgelöst mit dem Heading zum Zeitpunkt des Schrittes und einer festgelegten Schrittlänge.

In [4]:
df_steps = df_steps.drop(columns='heading_rad', errors='ignore') # drop heading_rad if already exists

df_steps = pd.merge_asof(
    df_steps,
    df_heading[['t_sec', 'heading_rad']],
    on='t_sec',
    direction='nearest'
)[['t_sec', 'heading_rad']]
df_steps['dt'] = np.diff(df_steps.t_sec, prepend=0.0)
df_steps.head(5)

MergeError: Incompatible merge dtype, dtype('O') and dtype('float64'), both sides must have numeric dtype

## Vorverarbeitung der RSSI Daten
Es wird ein Fenster über die Daten geschoben und die Messwerte aller Beacons pro ganze Sekunde zusammengefasst.

In [ ]:
window_size = 1000 # Sliding window size in ms

ble = df_rssi.copy()

ble["rssi"] = pd.to_numeric(ble["rssi"], errors="coerce")
ble["timestamp_ms"] = pd.to_numeric(ble["timestamp_ms"], errors="coerce")

ble = (
    ble
    .dropna(subset=["run_id", "timestamp_ms", "beacon_name", "rssi"])
    .query("rssi < 0")
    .copy()
)

ble["window_ms"] = (
    ble["timestamp_ms"] // window_size
).astype("int64") * window_size

rssi_windows = (
    ble
    .pivot_table(
        index=["run_id", "window_ms"],
        columns="beacon_name",
        values="rssi",
        aggfunc="median"
    )
    .sort_index()
    .reset_index()
)

rssi_windows["time_readable"] = pd.to_datetime(
    rssi_windows["window_ms"],
    unit="ms",
    utc=True
).dt.tz_convert("Europe/Berlin")

print("Beacons in der Datenbank:")
print(rssi_windows.columns.drop(
            ["run_id", "window_ms", "time_readable"]
        ).tolist()
)

rssi_windows['t_sec'] = (rssi_windows.window_ms - rssi_windows.window_ms.min()) / 1000

rssi_windows = rssi_windows[rssi_windows.run_id == run_id]
rssi_windows.head()

Beacons in der Datenbank:
['arrive_emi1', 'arrive_emi10', 'arrive_emi2', 'arrive_emi3', 'arrive_emi4', 'arrive_emi8']


beacon_name,run_id,window_ms,arrive_emi1,arrive_emi10,arrive_emi2,arrive_emi3,arrive_emi4,arrive_emi8,time_readable,t_sec
0,R1,1781612173000,NaN,NaN,NaN,-98.0,-66.5,NaN,2026-06-16 14:16:13+02:00,0.0
1,R1,1781612174000,NaN,NaN,NaN,-89.5,-74.0,NaN,2026-06-16 14:16:14+02:00,1.0
2,R1,1781612175000,NaN,NaN,NaN,-93.0,-82.0,NaN,2026-06-16 14:16:15+02:00,2.0
3,R1,1781612176000,NaN,NaN,NaN,-96.0,-80.0,NaN,2026-06-16 14:16:16+02:00,3.0
4,R1,1781612177000,NaN,NaN,NaN,-91.0,-79.0,NaN,2026-06-16 14:16:17+02:00,4.0


In [ ]:
RSSI_0           = -70.0  # dBm bei d=1m
PATH_LOSS_EXP    =   2.0  # Innenraum-Exponent
FLOOR_HEIGHT_M = 1.5  # dB Verlust pro Stockwerk
USE_OTHER_FLOOR = True  # True = andere Etagen mit Penalty, False = ignorieren

eg_beacons = [b for b, (_, _, f) in BEACON_POSITIONS.items() if f == 0]
og_beacons = [b for b, (_, _, f) in BEACON_POSITIONS.items() if f == 1]

def estimate_floor(rssi_window: pd.DataFrame) -> int:
    """Heuristik: Stärkere mittlere RSSI = wahrscheinlich selbes Stockwerk."""
    rssi_eg = rssi_window[rssi_window['beacon_name'].isin(eg_beacons)]['rssi'].mean()
    rssi_og = rssi_window[rssi_window['beacon_name'].isin(og_beacons)]['rssi'].mean()

    if pd.isna(rssi_eg) and pd.isna(rssi_og):
        return -1  # keine Beacons sichtbar

    # NaN als -inf behandeln → das andere Stockwerk gewinnt automatisch
    rssi_eg = rssi_eg if not pd.isna(rssi_eg) else float("-inf")
    rssi_og = rssi_og if not pd.isna(rssi_og) else float("-inf")

    return 0 if rssi_eg >= rssi_og else 1

def rssi_to_distance(rssi: float, floor_delta: int = 0) -> float:
    """
    Inverse Log-Distance Path Loss mit additivem Floor-Aufschlag.
    floor_delta=0 → gleiche Etage
    floor_delta>0 → +FLOOR_HEIGHT_M pro Etage wird addiert
    """
    if floor_delta != 0 and not USE_OTHER_FLOOR:
        return np.nan

    base_dist = 10.0 ** ((RSSI_0 - rssi) / (10.0 * PATH_LOSS_EXP))

    if floor_delta == 0:
        return base_dist

    # Fester Aufschlag pro Etage – unabhängig von base_dist
    return base_dist + abs(floor_delta) * FLOOR_HEIGHT_M


# ── Anwenden ────────────────────────────────────────────────────────
df_rssi['beacon_floor'] = df_rssi['beacon_name'].map(
    lambda b: BEACON_POSITIONS.get(b, (None, None, 0))[2]
)
df_rssi['floor_delta'] = df_rssi['beacon_floor'].fillna(0).astype(int)
df_rssi['dist_est_m']  =df_rssi.apply(
    lambda r: rssi_to_distance(r['rssi'], r['floor_delta']), axis=1
)

# Andere Etagen rausfiltern (wenn USE_OTHER_FLOOR=False)
rssi_df_clean = df_rssi.dropna(subset=['dist_est_m'])

print('Distanzschätzungen (Beispiel):')
print(rssi_df_clean[['run_id','beacon_name','rssi','floor_delta','dist_est_m']].head(10).round(2))

Distanzschätzungen (Beispiel):
  run_id  beacon_name  rssi  floor_delta  dist_est_m
0     R1  arrive_emi3 -98.0            1       26.62
1     R1  arrive_emi4 -66.0            0        0.63
2     R1  arrive_emi4 -67.0            0        0.71
3     R1  arrive_emi4 -66.0            0        0.63
4     R1  arrive_emi4 -72.0            0        1.26
5     R1  arrive_emi4 -78.0            0        2.51
6     R1  arrive_emi3 -91.0            1       12.72
7     R1  arrive_emi3 -84.0            1        6.51
8     R1  arrive_emi4 -76.0            0        2.00
9     R1  arrive_emi3 -92.0            1       14.09


## Plot
Hier wird die Funktion zum Plotten des aktuellen States des Particle Filters implementiert.

In [ ]:
H, W = map_eg.shape

def plot_map_particles(occupancy_map, particles, resolution=0.2):
    fig, ax = plt.subplots(figsize=(14, 14 * H / W + 1))

    # Map: extent in Metern, origin='lower' = row 0 unten (wie beim Plotly-Heatmap)
    ax.imshow(
        occupancy_map,
        cmap='gray',                      # 0 -> schwarz, 1 -> weiß (wie eure colorscale)
        origin='lower',
        extent=[0, W * resolution, 0, H * resolution],
        interpolation='nearest',
    )

    # Partikel
    sc = ax.scatter(
        particles[:, 0], particles[:, 1],
        s=3,
        c=particles[:, 2],
        cmap='viridis',
    )
    fig.colorbar(sc, ax=ax, label='Gewicht', shrink=0.6)

    ax.set_aspect('equal')
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    ax.set_title(f'{len(particles)} Partikel')
    plt.tight_layout()
    plt.show()

NameError: name 'map_eg' is not defined

## Particle Filter
### Initialisierung

Zu Beginn werden $m$ Partikel zufällig per Gleichverteilung in der Karte initialisiert

In [ ]:
def initialize_particles(occupancy_map, m=300, resolution=0.2):
    """
    Initializes m particles inside the occupancy map.
    :param occupancy_map: A binary map representing the building structure.
    :param m: The amount of particles to initialize.
    :param resolution: The resolution of the map.
    :returns: A mx2 matrix with the coordinates (x,y) of m particles.
    """
    free = np.argwhere(occupancy_map == 1)
    idx = np.random.default_rng().integers(0, len(free), m)
    cells = free[idx]

    x = (cells[:, 1] + np.random.default_rng().random(m)) * resolution
    y = (cells[:, 0] + np.random.default_rng().random(m)) * resolution

    particles = np.column_stack([x, y, np.ones((m,))])
    
    return particles

## Prediktions-Schritt
Im Prädiktionsschritt werden die Partikel basierend auf der PDR um einen Schritt Richtung des gemessenen Heading bewegt. Dabei wird ein Normalverteilter Fehler sowohl dem Heading als auch der Distanz hinzugefügt.

In [ ]:
def move(particles, d, heading, sigma_d, sigma_heading):
    """
    Moves each particle using the given distance d and heading with a normal ditributed error N(0,sigma_d) and N(0, sigma_heading) applied.
    :param particles: A mx3 matrix with m particles.
    :param d: The distance to move the pixels in m.
    :param heading: The direction of movement in rad.
    :param sigma_d: The variance of the error distribution of the distance.
    :param sigma_heading: The variance of the error distribution of the heading.
    :returns: A mx3 matrix with m updated particles.
    """
    m, _ = particles.shape
    
    # Calculate the directional and orientational errors
    epsilon_d = np.random.default_rng().normal(0, sigma_d, m)
    epsilon_heading = np.random.default_rng().normal(0, sigma_heading, m)
    
    particles_moved = particles.copy()
    
    # apply motion model
    particles_moved[:, 0] += np.cos(heading + epsilon_heading) * (d + epsilon_d)
    particles_moved[:, 1] += np.sin(heading + epsilon_heading) * (d + epsilon_d)
    
    return particles_moved

def update_weights(old_particles, particles, occupancy_map, resolution=0.2):
    """
    Checks for each particle, if the line between the old and the new one passes a wall in the map.
    If a particle passes a wall, its weight will be set to 0.
    :param old_particles: A mx3 matrix of m particle coordinates in pixels at timestep k-1.
    :param particles: A mx3 matrix of m particle coordinates in pixels at timestep k.
    :param occupancy_map: A binary map representing walls and walkable areas.
    :returns: A mx3 matrix of m particles with new weights applied.
    """
    height, width = occupancy_map.shape
    p0 = old_particles[:, :2] / resolution
    p1 = particles[:, :2] / resolution
    d = p1 - p0 
    seg_len = np.linalg.norm(d, axis=1)
    n_samples = max(2, int(np.ceil(seg_len.max() / 0.5)) + 1)    
    t = np.linspace(0.0, 1.0, n_samples)                  # (S,)

    pts = p0[:, None, :] + d[:, None, :] * t[None, :, None]   # (m, S, 2)
    cols = np.round(pts[..., 0]).astype(int)              # x -> col
    rows = np.round(pts[..., 1]).astype(int)              # y -> row

    # Out-of-bounds zählt als Wandtreffer
    oob = (rows < 0) | (rows >= H) | (cols < 0) | (cols >= W)
    r = np.clip(rows, 0, H - 1)
    c = np.clip(cols, 0, W - 1)

    hit = (occupancy_map[r, c] == 0) | oob       # (m, S)
    crossed = hit.any(axis=1)                             # (m,)

    particles[crossed, 2] = 0.0
    
    return particles

## Update-Schritt

In [ ]:
def update_weights_rssi(particles, measured_rssi, beacon_positions, sigma_dist=3.0):
    """
    Weights each particle by comparing the RSSI-derived distance estimate
    to the geometric distance from each particle to each beacon.
    """
    measured_rssi = np.asarray(measured_rssi, dtype=float)
    heard = ~np.isnan(measured_rssi)

    # RSSI -> Distanz über die bereits existierende Funktion
    measured_dist = np.array([rssi_to_distance(r) for r in measured_rssi[heard]])

    # Geometrische Distanz jedes Partikels zu jedem gehörten Beacon
    distances = np.linalg.norm(
        particles[:, None, :2] - beacon_positions[None, :, :], axis=2
    )
    distances = np.maximum(distances, 0.1)
    particle_dist = distances[:, heard]

    error = particle_dist - measured_dist[None, :]
    log_likelihood = -0.5 * np.sum((error / sigma_dist) ** 2, axis=1)

    particles[:, 2] *= np.exp(log_likelihood - log_likelihood.max())
    return particles


def resample(particles, position_jitter=0.15):
    """
    Systematic resampling: draws a new particle set from the current one,
    proportional to the weights. All weights are reset to 1 afterwards.
    """
    number_of_particles = len(particles)
    weights = particles[:, 2]

    total = weights.sum()
    if total < 1e-12:
        return particles          # filter collapsed, keep as is (or re-init globally)

    cumulative = np.cumsum(weights / total)
    positions = (np.arange(number_of_particles) + rng.random()) / number_of_particles
    chosen = np.searchsorted(cumulative, positions)

    resampled = particles[chosen].copy()
    resampled[:, :2] += rng.normal(0, position_jitter, (number_of_particles, 2))
    resampled[:, 2] = 1.0
    return resampled

In [ ]:
# Define beacon locations

beacon_names_order = [b for b in beacon_pos_df['beacon_name'].tolist()
                       if b in rssi_windows.columns]

beacon_positions_arr = (
    beacon_pos_df.set_index('beacon_name')
    .loc[beacon_names_order, ['x_m', 'y_m']]
    .to_numpy()
)

steps = len(df_steps)
d=0.7

step = 0
speed = 1.0

particles = initialize_particles(map_eg, m=300)

rssi_readings_index = 0

for _, row in df_steps.iterrows():
        t0 = time.perf_counter()
        current_timestep = row.t_sec

        # prediction step
        old_particles = particles.copy()
        particles = move(particles, d, df_steps.iloc[step].heading_rad + math.pi/2, 0.07, 0.05)
        particles = update_weights(old_particles, particles, map_eg)
        
        # update step
        if (current_timestep > rssi_windows.iloc[rssi_readings_index].t_sec):
            rssi_readings = [
                rssi_windows.iloc[rssi_readings_index].get(b, np.nan)
                for b in beacon_names_order
            ]
            
            particles = update_weights_rssi(
                particles, 
                rssi_readings, 
                beacon_positions_arr
            )
            particles = resample(particles)
            rssi_readings_index += 1

        clear_output(wait=True)
        plot_map_particles(map_eg, particles)

        elapsed = time.perf_counter() - t0
        wait = row['dt'] / speed - elapsed
        if wait > 0:
            time.sleep(wait)
            
        step += 1

KeyboardInterrupt: 

In [ ]:
rssi_windows

beacon_name,run_id,window_ms,arrive_emi1,arrive_emi10,arrive_emi2,arrive_emi3,arrive_emi4,arrive_emi8,time_readable,t_sec
0,R1,1781612173000,NaN,NaN,NaN,-98.0,-66.5,NaN,2026-06-16 14:16:13+02:00,0.0
1,R1,1781612174000,NaN,NaN,NaN,-89.5,-74.0,NaN,2026-06-16 14:16:14+02:00,1.0
2,R1,1781612175000,NaN,NaN,NaN,-93.0,-82.0,NaN,2026-06-16 14:16:15+02:00,2.0
3,R1,1781612176000,NaN,NaN,NaN,-96.0,-80.0,NaN,2026-06-16 14:16:16+02:00,3.0
4,R1,1781612177000,NaN,NaN,NaN,-91.0,-79.0,NaN,2026-06-16 14:16:17+02:00,4.0
...,...,...,...,...,...,...,...,...,...,...
56,R1,1781612229000,-98.5,NaN,NaN,NaN,NaN,-80.0,2026-06-16 14:17:09+02:00,56.0
57,R1,1781612230000,NaN,NaN,NaN,NaN,NaN,-93.0,2026-06-16 14:17:10+02:00,57.0
58,R1,1781612231000,NaN,NaN,NaN,NaN,NaN,-92.0,2026-06-16 14:17:11+02:00,58.0
59,R1,1781612233000,NaN,NaN,NaN,NaN,NaN,-76.0,2026-06-16 14:17:13+02:00,60.0
